# Degree-weighted metapath prediction of `Gene –reduce→ Disease` on the HALD knowledge graph



## 1. Background

This notebook implements an approach - degree-weighted path counts - to inference on graphs developed by Himmelstein & Baranzini in two papers:

* *Heterogeneous Network Edge Prediction: A Data Integration Approach to Prioritize Disease-Associated Genes*, PLOS Computational Biology, DOI:10.1371/journal.pcbi.1004259 July 9, 2015
* *Systematic integration of biomedical knowledge prioritizes drugs for repurposing*, Life, DOI:https://doi.org/10.7554/eLife.26726E, 2017

The approach is based on the concept of a *metapath*. Let's introduce this now.

### 1.1 HALD as a Heterogeneous network
HALD is a knowledge graph whose nodes belong to one of $K=10$ **node types** (classes): Gene, Disease, Mutation, RNA, Lipid, Carbohydrate, Peptide, Protein, Toxin and Pharmaceutical_Preparations. Nodes are linked by ~3,000 verb-like predicates (`associated`, `reduce`, `treated`, ...).

We define the network as a graph $G=(V,E)$ together with a map $\tau:V\to\{1,\dots,K\}$ from each node to one of the $K$ node types.

A **metaedge** is an ordered pair of node types $(a,b)$.

Because HALD has so many predicates, we **collapse all predicates** for simplicity and simply treat all edges as being the same: the metaedge $(a,b)$ contains every distinct edge $x\to y$ with $\tau(x)=a$, $\tau(y)=b$. For each metaedge we store a binary adjacency matrix

$$A_{ab}\in\{0,1\}^{n_a\times n_b},\qquad (A_{ab})_{xy}=1 \iff x\to y \text{ is an edge of metaedge } (a,b).$$

### 1.2 Metapaths

A **metapath** is a sequence of node types joined by metaedges, traversed in either direction:

$$m = \big(a_0 \xrightarrow{e_1} a_1 \xrightarrow{e_2} \cdots \xrightarrow{e_L} a_L\big),\qquad L=\text{length}.$$

A traversal "forward" along $(a,b)$ uses $A_{ab}$; "backward" uses $A_{ab}^{\top}$. In what follows, we will denote this as (for example) `Gene>Disease<Gene>Disease` which means: Gene→Disease (forward), then Disease←Gene (backward), then Gene→Disease (forward).

In this exercise we are going to consider metaedges of type `Gene - reduce - Disease`. Therefore, for our target, $a_0=\text{Gene}$ and $a_L=\text{Disease}$.

### 1.3 Target relation
We want to predict `Gene - reduce - Disease` of which there are 132 known edges. All of those edges are **removed** from the network before any feature is computed, so the target can never leak into a feature. Other Gene→Disease edges (e.g. `associated`) remain.

### 1.4 Degree-weighted path count (DWPC)
The plain **path count** $PC_m(s,t)$ is the number of paths from $s$ to $t$ that follow $m$. It is dominated by hub nodes. Himmelstein & Baranzini therefore chose to down-weight each path by the **metaedge-specific degrees** of the nodes on it.

For an edge $u\to v$ in metaedge $(a,b)$ define the out-degree of $u$ and in-degree of $v$ within that metaedge:

$$d^{\text{out}}_{ab}(u)=\sum_{v}(A_{ab})_{uv},\qquad d^{\text{in}}_{ab}(v)=\sum_{u}(A_{ab})_{uv}.$$

Each edge on a path contributes **two** degrees (one for each endpoint). Given a **damping exponent** $w\ge 0$, the **path-degree product** is

$$\mathrm{PDP}(\pi)=\prod_{i=1}^{L}\Big[d^{\text{out}}_{e_i}(u_{i-1})\Big]^{-w}\Big[d^{\text{in}}_{e_i}(u_i)\Big]^{-w}, \qquad \pi=(u_0,u_1,\dots,u_L),$$

and the **DWPC** is the sum over all paths from $s$ to $t$ following $m$ **with no repeated node**:

$$\boxed{\;\mathrm{DWPC}_m(s,t;w)=\sum_{\pi\in\Pi_m(s,t)}\mathrm{PDP}(\pi)\;}$$

* $w=0$ gives the ordinary path count.
* Larger $w$ penalises paths through high-degree nodes more strongly. The papers found $w\approx0.4$–$0.5$ best on their networks; we select it on HALD by cross-validation (section 6).

## 2. Load HALD and extract typed edges with SPARQL

We load the RDF/XML with OWLready2 and use SPARQL to pull out (i) the node lists per class, (ii) the positive `Gene –reduce→ Disease` pairs, and (iii) the edges of every metaedge.

**Edit the two paths in the next cell** to point at `HALD.rdf` and an output folder.

In [ ]:
import os, time, random
from collections import defaultdict
import numpy as np, pandas as pd, scipy.sparse as sp
from owlready2 import *
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# ------------------------------------------------------------------ configuration
RDF    = "./HALD_Dataset/HALD.rdf"          # path to the HALD RDF/XML file
OUT    = "results_dwpc"      # output folder (CSV results + SQLite cache)
MAXLEN = 3                   # maximum metapath length (3 = up to three metaedges)
N_PERM = 5                   # number of permuted networks
W_GRID = [0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.8, 1.0]   # damping exponents to try
SEED, N_NEG = 42, 20000      # random seed; number of sampled negative pairs

os.makedirs(OUT, exist_ok=True)
random.seed(SEED); rng = np.random.default_rng(SEED)


In [ ]:
# Load the ontology; the SQLite backend caches the parsed triples so later runs start quickly.
#default_world.set_backend(filename=os.path.join(OUT, "hald.sqlite3"))
onto = get_ontology("file://" + os.path.abspath(RDF)).load()
default_world.save()

# Define a lambda function that automatically prepends the namespace prefixes we will use.
PRE = ("PREFIX onto: <http://www.dummy.info/new.owl#>\n"
       "PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>\n"
       "PREFIX owl: <http://www.w3.org/2002/07/owl#>\n")
sparql = lambda q: default_world.sparql(PRE + q)

ClassCounts = list(sparql(
       """
       SELECT ?type (COUNT(DISTINCT ?node) AS ?count)
       WHERE {
              ?node a ?type
              ?type a owl:Class
       }
       GROUP BY ?type
       ORDER BY DESC(?count)
       """
))

# Extract just the classes for convenience
ClassList = [i[0] for i in ClassCounts]

# Index the nodes for later use in adjacency matrices
Nodes = dict()
for c in ClassList:
    nodelist = list(sparql(f"SELECT ?i WHERE {{ ?i a onto:{c.name} }}"))
    Nodes[c.name] = sorted(i[0].name for i in nodelist)
idx   = {c: {n: k for k, n in enumerate(v)} for c, v in Nodes.items()}
print(idx)
print(ClassCounts)
print(Nodes)

### Positive examples (the target relation)

The positive set is
$$\mathcal P=\{(g,d)\;:\; g\in\text{Gene},\ d\in\text{Disease},\ g\ \texttt{reduce}\ d\}.$$

In [ ]:
Targets = list(sparql(
    """
    SELECT ?gene ?disease
    WHERE {
        ?gene a onto:Gene
        ?disease a onto:Disease
        ?gene onto:reduce ?disease
    }
    """
    ))

print(Targets)

# Get the pairs sorted.
pos = sorted([(g.name, d.name) for g, d in Targets])
# Convert this to an index array - will be needed later during validation and testing
pos_idx = np.array([(idx["Gene"][g], idx["Disease"][d]) for g, d in pos])

print(len(pos), "positive Gene-reduce-Disease pairs")

### Metaedges

For every ordered pair of classes $(a,b)$ we run a SPARQL query returning the **distinct** pairs $(x,y)$ joined by *any* object property. Three choices matter:

* **Predicates are collapsed** (≈3,000 predicates would make the degrees extremely sparse).
* **Duplicate edges are merged**, so two different predicates between the same pair count as one edge.
* **Self-loops are dropped** and **all `Gene reduce Disease` edges are excluded** (`FILTER(?p != h:reduce)` on the Gene→Disease metaedge) to prevent target leakage.

In [ ]:
EDGES = {}     # (a, b) -> array of (row index in class a, column index in class b)

# For each pair of node types linked by an edge, get the edges
for a in ClassList:
    for b in ClassList:
        metaedges = list(sparql(
            f"""
            SELECT DISTINCT ?n1 ?n2 
            WHERE {{
                ?n1 a onto:{a.name}
                ?n2 a onto:{b.name}
                ?n1 ?p ?n2
                ?p a owl:ObjectProperty    
                FILTER( !( ?p = onto:reduce && onto:{a.name} = onto:Gene && onto:{b.name} = onto:Disease) )
            }}
            """
        ))

        # Build a list of edge pairs (nodes with an edge between them), removing self-edges and taking their index from the list we created earlier.
        edgepairs = {(idx[a.name][x.name], idx[b.name][y.name]) for x, y in metaedges if not (a.name == b.name and x.name == y.name)}
        # If there are some edges between a pair of types, put them in the master EDGES table. This is a dictionary keyed by a tuple of the two node types.
        if edgepairs:
            EDGES[(a.name, b.name)] = np.array(sorted(edgepairs))

# print some summary data
print(len(EDGES), "metaedges,", sum(len(e) for e in EDGES.values()), "distinct edges")
metaedge_table = pd.DataFrame([(a, b, len(e)) for (a, b), e in EDGES.items()],
                              columns=["src", "dst", "edges"]).sort_values("edges", ascending=False)
metaedge_table.to_csv(f"{OUT}/metaedges.csv", index=False)
metaedge_table.head(10)

### Adjacency matrices and a SPARQL cross-check of the degrees

`adjacency` builds the sparse binary matrices $A_{ab}$. As a sanity check, we compare the metaedge-specific degree of the best-connected gene computed from the matrix with the same quantity computed directly by a SPARQL `COUNT(DISTINCT ...)`.

In [ ]:
# For each *type* of node-node edge calculate the total number of metaedge between two nodes of that type
# Example: A0[('Gene', 'Disease')] holds
# Coords	Values
# (5, 288)	1.0
# (5, 836)	1.0
# (5, 1360)	1.0
# (5, 3465)	1.0
# (11, 123)	1.0
# (12, 1844)	1.0
# (12, 1925)	1.0
#
# which means that node 5 of type Gene has 1 connection to node 1360 of type disease etc etc
print(Nodes)


def adjacency(edges,nodes):
    X = dict()
    for k,e in edges.items():
        X[k] = sp.csr_matrix((np.ones(len(e)), (e[:, 0], e[:, 1])), shape=(len(Nodes[k[0]]), len(Nodes[k[1]])))
    return X


A0 = adjacency(EDGES, Nodes)

# Fetch the Gene from the table with the highest connectivity to Disease
g0 = Nodes["Gene"][int(np.argmax(np.asarray(A0[("Gene", "Disease")].sum(1))))]


# Get that node by name and count its connections, removing reduce connections
q = list(sparql(
    f'''
    SELECT (COUNT(DISTINCT ?d) AS ?n)
    WHERE {{ 
        onto:{g0} ?p ?d
        ?d rdf:type onto:Disease
        ?p a owl:ObjectProperty
        FILTER(?p != onto:reduce)
    }}
    '''
))[0][0]

# If the counts returned by the two methods don't match raise an assertion
assert q == int(A0[("Gene", "Disease")][idx["Gene"][g0]].sum()), "degree mismatch"
print(f"degree check OK (Node {g0}: Gene>Disease degree = {q})")

## 3. Enumerate metapaths

Starting at Gene, we follow every metaedge in either direction (`>` forward, `<` backward) for up to `MAXLEN` steps, and keep every path that ends at Disease. The number of metapaths grows quickly with length:

$$|\mathcal M_{\le L}|\approx \sum_{\ell=1}^{L}\ \#\{\text{typed walks of length }\ell\text{ from Gene to Disease}\}.$$

In [ ]:
STEPS = {}     # class -> list of (next class, direction, metaedge key)

# Set up the rules for the metapath. For each edge (a,b) the allowabvle moves are a > b or b < a
# This is effectively a linked list.
for (a, b) in EDGES:
    STEPS.setdefault(a, []).append((b, ">", (a, b)))   # forward  : uses A_ab
    STEPS.setdefault(b, []).append((a, "<", (a, b)))   # backward : uses A_ab^T

print(STEPS)

# construct the paths recursively
def enumerate_metapaths(maxlen):
    out = []
    def rec(path):
        if len(path) > 1 and path[-1][0] == "Disease": out.append(tuple(path))
        if len(path) - 1 == maxlen: return
        for nxt, d, key in STEPS.get(path[-1][0], []):
            rec(path + [(nxt, d, key)])
    rec([("Gene", "", None)])
    return out

name_of = lambda mp: mp[0][0] + "".join(d + c for c, d, _ in mp[1:])
METAPATHS = {name_of(m): m for m in enumerate_metapaths(MAXLEN)}
print(len(METAPATHS), f"metapaths (length <= {MAXLEN}; both edge directions allowed)")
list(METAPATHS)[:10]

## 4. Computing the DWPC with sparse matrices

Enumerating paths one by one is expensive. Because the path-degree product factorises over edges, the DWPC for **all** $(s,t)$ pairs is a product of **weighted adjacency matrices**. For a metaedge $(a,b)$ define

$$W_{ab}(w)=D^{\text{out}}_{ab}(w)\; A_{ab}\; D^{\text{in}}_{ab}(w),\qquad
D^{\text{out}}_{ab}=\operatorname{diag}\!\big(d^{\text{out}}_{ab}(u)^{-w}\big),\quad
D^{\text{in}}_{ab}=\operatorname{diag}\!\big(d^{\text{in}}_{ab}(v)^{-w}\big).$$

so $(W_{ab})_{uv}=d^{\text{out}}(u)^{-w}\,d^{\text{in}}(v)^{-w}$ if $u\to v$ is an edge. Writing $M_i=W_{e_i}$ for forward steps and $M_i=W_{e_i}^{\top}$ for backward steps,

$$\mathrm{DWPC}^{\text{walk}}_m(w)=M_1M_2\cdots M_L .$$

This matrix counts **walks**, which may revisit nodes. Himmelstein & Baranzini exclude paths with duplicate nodes. Adjacent repeats ($u_i=u_{i+1}$) are impossible because self-loops were removed. For $L=3$, with nodes $(n_0,n_1,n_2,n_3)$, the remaining possible repeats are $n_0=n_2$ and $n_1=n_3$ ($n_0=n_3$ is impossible since $n_0$ is a Gene and $n_3$ a Disease). By inclusion–exclusion,

$$\mathrm{DWPC}_m = M_1M_2M_3 \;-\; \underbrace{\operatorname{diag}\!\big[(M_1\circ M_2^{\top})\mathbf 1\big]\,M_3}_{n_0=n_2\ (\text{if }a_0=a_2)}
\;-\; \underbrace{M_1\operatorname{diag}\!\big[(M_2\circ M_3^{\top})\mathbf 1\big]}_{n_1=n_3\ (\text{if }a_1=a_3)}
\;+\; \underbrace{M_1\circ M_2^{\top}\circ M_3}_{\text{both (only for Gene–Disease–Gene–Disease)}}$$

where $\circ$ is the element-wise product. For $L\le2$ no correction is needed (the end types are Gene and Disease, so only adjacent repeats could occur).

In [ ]:
def weighted(A, w):
    # W_ab = D_out^-w  A_ab  D_in^-w  for every metaedge
    W = {}
    for k, M in A.items():
        out_d = np.asarray(M.sum(1)).ravel() # ravel flattens the array
        in_d = np.asarray(M.sum(0)).ravel()
        do = sp.diags(np.where(out_d > 0, out_d, 1.0) ** -w)
        di = sp.diags(np.where(in_d  > 0, in_d,  1.0) ** -w)
        W[k] = (do @ M @ di).tocsr()
    return W

def dwpc_matrix(mp, W):
    # full (Gene x Disease) DWPC matrix for one metapath
    mats  = [W[key] if d == ">" else W[key].T.tocsr() for _, d, key in mp[1:]]
    types = [c for c, _, _ in mp]
    M = mats[0]
    for X in mats[1:]: M = M @ X
    if len(mats) == 3:                                   # remove paths with duplicate nodes
        W1, W2, W3 = mats
        a_eq = types[0] == types[2]
        b_eq = types[1] == types[3]
        if a_eq:
            M = M - sp.diags(np.asarray(W1.multiply(W2.T).sum(1)).ravel()) @ W3
        if b_eq:
            M = M - W1 @ sp.diags(np.asarray(W2.multiply(W3.T).sum(1)).ravel())
        if a_eq and b_eq:
            M = M + W1.multiply(W2.T).multiply(W3)
    return M.tocsr()

def features(A, w, names, pairs):
    # DWPC value of each metapath in `names` for each (gene, disease) pair in `pairs`
    W = weighted(A, w); cols = []
    for n in names:
        M = dwpc_matrix(METAPATHS[n], W)
        cols.append(np.asarray(M[pairs[:, 0], pairs[:, 1]]).ravel())
    return np.column_stack(cols)

### Verification against brute-force path enumeration

To check the matrix implementation, `brute` walks the graph path by path, multiplies the degree terms $d^{-w}$ along the way, skips repeated nodes, and sums. The two must agree to numerical precision.

In [ ]:
def brute(A, mp, w, g, d):
    deg = {k: (np.asarray(M.sum(1)).ravel(), np.asarray(M.sum(0)).ravel()) for k, M in A.items()}
    total = 0.0
    def rec(i, node, pdp, seen):
        nonlocal total
        if i == len(mp):
            if node == d: total += pdp
            return
        c, dirn, key = mp[i];
        M = A[key] if dirn == ">" else A[key].T.tocsr()
        od, idg = deg[key] if dirn == ">" else deg[key][::-1]
        for nb in M[node].indices:
            if (c, nb) in seen: continue
            rec(i + 1, nb, pdp * od[node] ** -w * idg[nb] ** -w, seen | {(c, nb)})
    rec(1, g, 1.0, {("Gene", g)})
    return total

for chk in ["Gene>Disease>Disease", "Gene>Disease<Gene>Disease"]:     # length 2 and a length-3 path with repeats
    if chk in METAPATHS:
        Mc = dwpc_matrix(METAPATHS[chk], weighted(A0, 0.4))
        for g, d in pos_idx[:3]:
            assert abs(Mc[g, d] - brute(A0, METAPATHS[chk], 0.4, g, d)) < 1e-9
        print(f"{chk}: matrix DWPC matches brute-force enumeration")

## 5. Training data: positives and sampled negatives

* **Positives** $\mathcal P$: the 132 `Gene reduce Disease` pairs.
* **Negatives**: `N_NEG` = 20,000 gene–disease pairs sampled uniformly from all pairs not in $\mathcal P$. (There are $5624\times3501\approx19.7$ M such pairs; performance is limited by the small number of positives, not negatives.)

### Feature transformation
DWPCs are heavily right-skewed (long paths much more common) so each feature is **mean-scaled and passed through the inverse hyperbolic sine**, then **standardised to z-scores**:

$$\tilde x=\operatorname{arcsinh}\!\Big(\frac{x}{\bar x}\Big),\qquad
z=\frac{\tilde x-\mu_{\tilde x}}{\sigma_{\tilde x}},\qquad \operatorname{arcsinh}(u)=\ln\!\big(u+\sqrt{u^2+1}\big).$$

($\bar x$ is the training-set mean of the feature.) $\operatorname{arcsinh}$ behaves like $u$ near 0 and like $\ln 2u$ for large $u$, and, unlike $\log$, handles zeros. Standardising makes the regression coefficients comparable across features.

### Evaluation metrics
For scores $s$ and labels $y$:
* **AUROC** $=P(s_{\text{pos}}>s_{\text{neg}})$ for a random positive/negative pair.
* **Average precision** $\mathrm{AP}=\sum_k (R_k-R_{k-1})P_k$, the area under the precision–recall curve. With only ~0.7 % positives, AP is far more sensitive to the top of the ranking than AUROC.

In [ ]:
n_g, n_d = len(Nodes["Gene"]), len(Nodes["Disease"])
posset = {tuple(p) for p in pos_idx.tolist()}
neg = set()
while len(neg) < N_NEG:
    p = (int(rng.integers(n_g)), int(rng.integers(n_d)))
    if p not in posset:
        neg.add(p)

PAIRS = np.vstack([pos_idx, np.array(sorted(neg))])                       # (gene index, disease index)
Y = np.r_[np.ones(len(pos_idx)), np.zeros(N_NEG)].astype(int)
print(PAIRS.shape[0], "pairs:", Y.sum(), "positive,", (Y == 0).sum(), "negative")

class MeanArcsinh(BaseEstimator, TransformerMixin):
    # mean scaling followed by inverse hyperbolic sine (Project Rephetio)
    def fit(self, X, y=None):
        m = X.mean(0); self.m_ = np.where(m > 0, m, 1.0); return self
    def transform(self, X): return np.arcsinh(X / self.m_)

def auc_table(X, names):
    # single-feature AUROC / AP, for features connected to >= 5 pairs
    rows = []
    for j, n in enumerate(names):
        if (X[:, j] > 0).sum() >= 5 and X[Y == 1, j].any():
            rows.append((n, roc_auc_score(Y, X[:, j]), average_precision_score(Y, X[:, j]), int((X[Y == 1, j] > 0).sum())))
    return pd.DataFrame(rows, columns=["metapath", "AUC", "AP", "pos_covered"]).set_index("metapath")

## 6. Choosing the damping exponent $w$

The papers treat $w$ as the single DWPC hyper-parameter and optimise it by cross-validated model performance. Here, for every $w$ in the grid, we compute DWPCs for all metapaths of length $\le 2$, fit a ridge-regularised logistic regression (below), and score **5-fold cross-validated out-of-fold AUROC**. We pick the $w$ with the largest AUROC.

### The model
Logistic regression with an $\ell_2$ (ridge) penalty, as selected by Himmelstein & Baranzini, models

$$P(y=1\mid \mathbf z)=\sigma(\beta_0+\boldsymbol\beta^{\top}\mathbf z),\qquad \sigma(u)=\frac1{1+e^{-u}},$$

by minimising the class-weighted penalised negative log-likelihood

$$\hat{\boldsymbol\beta}=\arg\min_{\boldsymbol\beta}\ \ \frac{1}{2}\|\boldsymbol\beta\|_2^2\;+\;C\sum_{i}c_{y_i}\Big[-y_i\log p_i-(1-y_i)\log(1-p_i)\Big],$$

where $c_y$ balances the classes ($c_1\gg c_0$) and $C$ is the inverse regularisation strength (small $C$ = strong shrinkage).

In [ ]:
def make_ridge(C=0.01):
    return make_pipeline(MeanArcsinh(), StandardScaler(),
                         LogisticRegression(C=C, class_weight="balanced", max_iter=3000))

def cv_oof(Xm, mk=make_ridge):
    # 5-fold stratified CV; return out-of-fold AUROC and AP
    oof = np.zeros(len(Y))
    for tr, te in StratifiedKFold(5, shuffle=True, random_state=SEED).split(Xm, Y):
        oof[te] = mk().fit(Xm[tr], Y[tr]).predict_proba(Xm[te])[:, 1]
    return roc_auc_score(Y, oof), average_precision_score(Y, oof)

short = [n for n, m in METAPATHS.items() if len(m) <= 3]      # metapaths with <= 2 metaedges
wrows = []
for w in W_GRID:
    Xs = features(A0, w, short, PAIRS); Xs = Xs[:, (Xs > 0).sum(0) >= 5]
    a, p = cv_oof(Xs)
    t = auc_table(Xs, [f"m{i}" for i in range(Xs.shape[1])])
    wrows.append((w, a, p, t.AUC.mean()))
    print(f"w={w}: CV AUC={a:.4f}  AP={p:.4f}  mean single-metapath AUC={t.AUC.mean():.4f}")

wdf = pd.DataFrame(wrows, columns=["w", "CV_AUC", "CV_AP", "mean_single_AUC"])
wdf.to_csv(f"{OUT}/damping_exponent_grid.csv", index=False)
W_BEST = float(wdf.loc[wdf.CV_AUC.idxmax(), "w"])
print("selected damping exponent w =", W_BEST)
wdf

**Reading the table.** AUROC is essentially flat for $w\in[0,0.6]$ and drops at $w=1$. Average precision *decreases* as $w$ increases. This is a property of HALD plus the evaluation design (random negatives make hubs easy to detect, so removing the hub signal costs ranking precision). We follow the papers and select by AUROC.

## 7. Feature matrix and degree covariates

We compute the DWPC of **all** metapaths at the chosen $w$ (`X`) and at $w=0$ (`X0`, the plain path count) for the baseline comparison, and keep metapaths connecting at least 5 of our pairs.

### Degree covariates
We can also include **degree features**: the metaedge-specific degrees of the source and target node,

$$d^{\text{out}}_{e}(s),\ d^{\text{in}}_{e}(s)\ \ (\text{for the gene } s),\qquad d^{\text{out}}_{e}(t),\ d^{\text{in}}_{e}(t)\ \ (\text{for the disease } t),$$

for every metaedge $e$ touching Gene or Disease. They capture how "well connected" each node is, so the metapath features need only explain what is left. Because the permuted networks (section 8) preserve these degrees, comparing models *with and without* them is informative.

In [ ]:
NAMES = list(METAPATHS)
X  = features(A0, W_BEST, NAMES, PAIRS)      # DWPC at the selected w
X0 = features(A0, 0.0,    NAMES, PAIRS)      # plain path count (w = 0)
keep = [j for j in range(len(NAMES)) if (X[:, j] > 0).sum() >= 5]
NAMES = [NAMES[j] for j in keep]; X, X0 = X[:, keep], X0[:, keep]
print(len(NAMES), "metapaths with >= 5 connected pairs")

single  = auc_table(X, NAMES)
single["AUC_pathcount_w0"] = auc_table(X0, NAMES).AUC

def degree_features(A, pairs):
    cols, nm = [], []
    for (a, b), M in A.items():
        if a == "Gene":    cols.append(np.asarray(M.sum(1)).ravel()[pairs[:, 0]]); nm.append(f"deg_Gene_out_{a}>{b}")
        if b == "Gene":    cols.append(np.asarray(M.sum(0)).ravel()[pairs[:, 0]]); nm.append(f"deg_Gene_in_{a}>{b}")
        if a == "Disease": cols.append(np.asarray(M.sum(1)).ravel()[pairs[:, 1]]); nm.append(f"deg_Disease_out_{a}>{b}")
        if b == "Disease": cols.append(np.asarray(M.sum(0)).ravel()[pairs[:, 1]]); nm.append(f"deg_Disease_in_{a}>{b}")
    return np.column_stack(cols).astype(float), nm

XD, DNAMES = degree_features(A0, PAIRS)
kd = XD.std(0) > 0; XD = XD[:, kd]; DNAMES = [n for n, k in zip(DNAMES, kd) if k]
print(XD.shape[1], "degree covariates")

## 8. Degree-preserving permuted networks (null model)

Is a metapath informative *beyond* node degrees? Both papers answer this with **permuted networks**. Within each metaedge, edges are randomly rewired by repeated **double edge swaps**

$$(a\to b),\ (c\to d)\ \longmapsto\ (a\to d),\ (c\to b),$$

(rejecting swaps that would create duplicate edges or self-loops). Every node keeps its metaedge-specific degree, but the *specific* connections are randomised. A metapath with real signal should score a higher AUROC on the real network than on permuted ones.

For metapath $m$, with real AUROC $\mathrm{AUC}_m$ and permuted AUROCs $\mathrm{AUC}^{(1)}_m,\dots,\mathrm{AUC}^{(R)}_m$:

$$z_m=\frac{\mathrm{AUC}_m-\overline{\mathrm{AUC}^{\text{perm}}_m}}{\operatorname{sd}\big(\mathrm{AUC}^{\text{perm}}_m\big)},\qquad
p_m=\frac{1+\#\{r:\mathrm{AUC}^{(r)}_m\ge \mathrm{AUC}_m\}}{1+R}.$$

With $R=5$ permutations the smallest attainable empirical $p$ is $1/6\approx0.17$. Increase `N_PERM` for finer p-values.

In [ ]:
def permute(edges, n_rounds=10):
    out = {}
    for k, e in edges.items():
        e = e.copy(); S = {tuple(r) for r in e.tolist()}; same = k[0] == k[1]; m = len(e)
        for _ in range(n_rounds * m):
            i, j = rng.integers(m), rng.integers(m)
            (a, b), (c, d) = e[i], e[j]
            if i == j or (a, d) in S or (c, b) in S or (same and (a == d or c == b)): continue
            S.discard((a, b)); S.discard((c, d)); S.add((a, d)); S.add((c, b)); e[i] = (a, d); e[j] = (c, b)
        out[k] = e
    return out

perm_auc, perm_X = [], []
for r in range(N_PERM):
    Ap = adjacency(permute(EDGES),Nodes)
    Xp = features(Ap, W_BEST, NAMES, PAIRS); perm_X.append(Xp)
    perm_auc.append(auc_table(Xp, NAMES).AUC.reindex(NAMES))
    print(f"permuted network {r+1}/{N_PERM} done")

P = pd.concat(perm_auc, axis=1)
single["AUC_perm_mean"] = P.mean(1); single["AUC_perm_sd"] = P.std(1)
single["z_vs_perm"] = (single.AUC - single.AUC_perm_mean) / single.AUC_perm_sd.replace(0, np.nan)
single["p_empirical"] = [(1 + (P.loc[n] >= single.AUC[n]).sum()) / (1 + N_PERM) for n in single.index]
single = single.sort_values("AUC", ascending=False)
single.to_csv(f"{OUT}/metapath_performance_dwpc.csv")

### Single-metapath performance
**Top metapaths by raw AUROC** and **top metapaths by AUROC gain over the permuted null**. Long chains of Disease–Disease edges score highly in raw AUROC, but their permuted-network AUROC is *higher* — they mainly reflect disease hubness. The gain over the null is the better indicator of real structure.

In [ ]:
cols = ["AUC", "AP", "pos_covered", "AUC_pathcount_w0", "AUC_perm_mean", "z_vs_perm"]
print("Top 12 by raw AUROC"); display(single[cols].head(12).round(3))
single["gain_vs_perm"] = single.AUC - single.AUC_perm_mean
print("Top 12 by AUROC gain over permuted networks")
display(single.sort_values("gain_vs_perm", ascending=False)[cols + ["p_empirical"]].head(12).round(3))

## 9. Regularised logistic regression and cross-validation

We combine all metapath DWPCs (optionally with the degree covariates) in one ridge logistic regression (section 6). The regularisation strength $C$ is chosen once, by 5-fold CV on the DWPC + degree feature set. Because this uses the same folds that are then reported, the final numbers for that feature set are slightly optimistic; all feature sets use the same $C$, so the comparison between them is fair.

Feature sets compared:

| Feature set | Purpose |
|---|---|
| degree covariates only | how much is explained by node degree alone |
| DWPC ($w$ selected) | metapath signal |
| DWPC + degree | metapath signal on top of degree (the Rephetio model) |
| path count ($w=0$) (+ degree) | does degree-weighting help? |
| DWPC on a permuted network (+ degree) | null model for the full classifier |

In [ ]:
mkC = lambda C: (lambda: make_ridge(C))
cgrid = []
for C in [0.0001, 0.0003, 0.001, 0.003, 0.01]:
    a, p = cv_oof(np.hstack([X, XD]), mkC(C)); cgrid.append((C, a, p))
    print(f"C={C}: CV AUC={a:.3f}  AP={p:.3f}")
cdf = pd.DataFrame(cgrid, columns=["C", "CV_AUC", "CV_AP"]); cdf.to_csv(f"{OUT}/ridge_C_grid.csv", index=False)
C_BEST = float(cdf.loc[cdf.CV_AUC.idxmax(), "C"]); mk = mkC(C_BEST)
print("selected C =", C_BEST)

In [ ]:
summary = []
for label, Xm in [("degree features only", XD),
                  (f"DWPC w={W_BEST}", X),
                  (f"DWPC w={W_BEST} + degree", np.hstack([X, XD])),
                  ("path count (w=0)", X0),
                  ("path count (w=0) + degree", np.hstack([X0, XD])),
                  (f"DWPC w={W_BEST}, permuted net 1", perm_X[0]),
                  (f"DWPC w={W_BEST}, permuted net 1 + degree", np.hstack([perm_X[0], XD]))]:
    a, p = cv_oof(Xm, mk); summary.append((label, a, p))
    print(f"{label:42s} CV AUC={a:.3f}  AP={p:.3f}")
summary_df = pd.DataFrame(summary, columns=["features", "CV_AUC", "CV_AP"])
summary_df.to_csv(f"{OUT}/model_cv_summary.csv", index=False)
summary_df.round(3)

### Standardised coefficients
Because every feature is a z-score, the coefficients $\hat\beta_j$ are directly comparable "effect sizes" (change in log-odds per standard deviation of the feature). Fitting on all data:

In [ ]:
fin = mk().fit(np.hstack([X, XD]), Y)
coefs = pd.DataFrame({"feature": NAMES + DNAMES, "std_coef": fin[-1].coef_[0]}).sort_values("std_coef", ascending=False)
coefs.to_csv(f"{OUT}/model_standardised_coefficients.csv", index=False)
print("Most positive"); display(coefs.head(10).round(4))
print("Most negative"); display(coefs.tail(5).round(4))

## 10. Interpreting the results

* **Degree dominates.** Because negatives are random pairs, hub genes and diseases are easy to separate; degree covariates alone give a very high AUROC. The metapath features add modest gains, mostly in average precision.
* **Does degree-weighting help?** Compare "DWPC" with "path count (w=0)". The difference is small on HALD; the damping exponent matters little here, unlike in the sparser curated networks used in the papers.
* **Real structure beyond degree.** Metapaths such as `Gene>Disease` (other predicates linking the gene to the disease) and `Gene>Gene>Disease` clearly beat their permuted-network null. Long Disease–Disease chains do not.
* **Permuted networks.** Models trained on permuted features still do well because the permutations preserve degree, which is exactly why the papers use them to separate "degree" from "topology" effects.

### Caveats and extensions
* Only 132 positives, so cross-validated metrics are noisy.
* $w$ was chosen using length-$\le2$ metapaths only (for speed).
* Metaedges ignore predicate identity. A finer treatment (e.g. grouping predicates into semantic classes) could give more specific features.
* Negatives are uniform random pairs. **Degree-matched negatives** would give a fairer test of the topological signal.

**References.** Himmelstein DS & Baranzini SE (2015) *Heterogeneous network edge prediction: a data integration approach to prioritize disease-associated genes.* PLoS Comput Biol 11(7):e1004259.
Himmelstein DS et al. (2017) *Systematic integration of biomedical knowledge prioritizes drugs for repurposing.* eLife 6:e26726.